Lounes Advancement - Week 1 


Make sure you have all the following librairies:

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report)
import joblib

print('All libraries loaded OK')

All libraries loaded OK


1. Loading Data

In [3]:
TRAIN_PATH = 'data/UNSW_NB15_training-set/UNSW_NB15_training-set.csv'
TEST_PATH  = 'data/UNSW_NB15_testing-set/UNSW_NB15_testing-set.csv'

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)


Some Little Analysis concerning the global form:

In [4]:
#Basic information concerning the dataset:
print("Training DataSet info:")
print(train.info())

print("-----------------")

print("Testing DataSet info:")
print(test.info())

Training DataSet info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82332 entries, 0 to 82331
Data columns (total 45 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 82332 non-null  int64  
 1   dur                82332 non-null  float64
 2   proto              82332 non-null  object 
 3   service            82332 non-null  object 
 4   state              82332 non-null  object 
 5   spkts              82332 non-null  int64  
 6   dpkts              82332 non-null  int64  
 7   sbytes             82332 non-null  int64  
 8   dbytes             82332 non-null  int64  
 9   rate               82332 non-null  float64
 10  sttl               82332 non-null  int64  
 11  dttl               82332 non-null  int64  
 12  sload              82332 non-null  float64
 13  dload              82332 non-null  float64
 14  sloss              82332 non-null  int64  
 15  dloss              82332 non-null  int64  
 16 

## UNSW-NB15 — Dataset Description

Each row = one network flow. 49 features, 2 splits: 175,341 train / 82,332 test.

### Target
| Column | Description |
|--------|-------------|
| `label` | 0 = benign · 1 = attack |
| `attack_cat` | Attack type name — **dropped before training** |

### Group 1 — Basic flow
| Column | Description |
|--------|-------------|
| `dur` | Flow duration (seconds) |
| `proto` | Protocol: tcp, udp, arp… |
| `service` | App service: http, ftp, dns… (`-` if none) |
| `state` | Connection state: FIN, CON, REQ, INT, RST |
| `spkts` / `dpkts` | Packets sent by source / destination |
| `sbytes` / `dbytes` | Bytes sent by source / destination |
| `rate` | Packets per second |
| `sttl` / `dttl` | Time-To-Live source / destination |
| `sload` / `dload` | Bits per second source / destination |
| `sloss` / `dloss` | Packets retransmitted or dropped |
| `sinpkt` / `dinpkt` | Inter-packet arrival time (ms) |
| `sjit` / `djit` | Jitter — variance in arrival time (ms) |
| `swin` / `dwin` | TCP window size |
| `stcpb` / `dtcpb` | TCP base sequence number |
| `smean` / `dmean` | Mean packet size |
| `trans_depth` | HTTP pipeline depth |
| `response_body_len` | HTTP response body size |

### Group 2 — TCP timing
| Column | Description |
|--------|-------------|
| `tcprtt` | Round-trip time (seconds) |
| `synack` | Time between SYN and SYN-ACK |
| `ackdat` | Time between SYN-ACK and ACK |

### Group 3 — Content
| Column | Description |
|--------|-------------|
| `is_sm_ips_ports` | 1 if source and destination IP/port are identical |
| `ct_state_ttl` | Count of flows with same state + TTL |
| `ct_flw_http_mthd` | Count of HTTP methods in flow |
| `is_ftp_login` | 1 if FTP session includes login |
| `ct_ftp_cmd` | Number of FTP commands in session |

### Group 4 — Aggregated counters (ct_)
Most powerful for detecting scans and brute-force. Count repeated behaviour over last 100 records.

| Column | Description |
|--------|-------------|
| `ct_srv_src` | Same source IP + same service |
| `ct_dst_ltm` | Same destination IP |
| `ct_src_dport_ltm` | Same source → same destination port |
| `ct_dst_sport_ltm` | Same destination ← same source port |
| `ct_dst_src_ltm` | Same source ↔ destination pair |
| `ct_src_ltm` | Same source IP |
| `ct_srv_dst` | Same destination IP + same service |

### Dropped columns
| Column | Reason |
|--------|--------|
| `id` | Row identifier, no predictive value |
| `attack_cat` | Leaks the target |

Further Analysis:

In [12]:
print('Nature of columns:')
print(train.dtypes.value_counts())

print('\n How many Label = 1 and how many label = 0')
print(train['label'].value_counts())
print(train['label'].value_counts(normalize=True).round(3))

Nature of columns:
int64      30
float64    11
object      4
Name: count, dtype: int64

 How many Label = 1 and how many label = 0
label
1    45332
0    37000
Name: count, dtype: int64
label
1    0.551
0    0.449
Name: proportion, dtype: float64


In order to well understand the topic and the main goal, Here is a description of how we are going to spot the different attacks thanks to the boundtiful dataset we have in possession: 

Models learn statistical patterns separating normal traffic from attacks across five main signals:

Signal 1 — Volume & rate: Attack traffic (DoS, Fuzzers) sends abnormally high packet/byte counts (spkts, sbytes) in short durations, pushing rate far above benign baselines.

Signal 2 — Connection state: Normal flows complete the TCP handshake and end in FIN/CON. Attack flows frequently stall in incomplete states (REQ, INT).

Signal 3 — TTL values: Each OS produces characteristic TTL values. Attackers spoofing their origin generate anomalous sttl/dttl combinations outside normal ranges.

Signal 4 — Jitter & timing: Automated attack tools produce unnaturally regular inter-packet intervals (sinpkt, dinpkt) and near-zero jitter (sjit) — unlike organic human/server traffic.

Signal 5 — Aggregated counters (ct_): The most powerful signal. Features like ct_srv_src count repeated connections from the same source, flagging scans and brute-force attempts that no normal user would generate.

Why this is hard: UNSW-NB15 has significant class overlap — some attack flows are statistically indistinguishable from benign traffic. This is why Recall on attacks is our primary metric: a missed attack is costlier than a false alarm.

In [13]:
def class_stats(df, name):
    total = len(df)
    benign = (df['label'] == 0).sum()
    attack = (df['label'] == 1).sum()
    return {
        'Split': name,
        'Total samples': total,
        'Benign (0)': benign,
        'Attack (1)': attack,
        'Benign %': f'{benign/total*100:.1f}%',
        'Attack %': f'{attack/total*100:.1f}%'
    }

table1 = pd.DataFrame([class_stats(train, 'Train'), class_stats(test, 'Test')])
table1.loc[2] = ['Total', len(train)+len(test),
                 (train['label']==0).sum()+(test['label']==0).sum(),
                 (train['label']==1).sum()+(test['label']==1).sum(), '-', '-']
table1.index = ['', '', '']

table1

,Split,Total samples,Benign (0),Attack (1),Benign %,Attack %
,Train,82332,37000,45332,44.9%,55.1%
,Test,175341,56000,119341,31.9%,68.1%
,Total,257673,93000,164673,-,-


We delete the "answering columns" and we proceed to a data cleaning:

In [14]:
# Colonnes à supprimer (identifiants, cible catégorielle)
DROP_COLS = ['id', 'attack_cat']
TARGET = 'label'

def preprocess(df):
    df = df.copy()
    df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)
    # Encoder les colonnes catégorielles (proto, service, state)
    for col in df.select_dtypes(include='object').columns:
        if col != TARGET:
            df[col] = LabelEncoder().fit_transform(df[col].astype(str))
    return df

train_clean = preprocess(train)
test_clean  = preprocess(test)

X_train = train_clean.drop(columns=[TARGET])
y_train = train_clean[TARGET]
X_test  = test_clean.drop(columns=[TARGET])
y_test  = test_clean[TARGET]

# Normalisation (pour LR surtout)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Features : {X_train.shape[1]}')
print('Preprocessing OK')

Features : 42
Preprocessing OK


Training:

In [15]:
ratio = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=100, scale_pos_weight=ratio, random_state=42,
                                         use_label_encoder=False, eval_metric='logloss', verbosity=0)
}

results = {}
for name, model in models.items():
    print(f'Training {name}...')
    X_tr = X_train_sc if name == 'Logistic Regression' else X_train
    X_te = X_test_sc  if name == 'Logistic Regression' else X_test
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    results[name] = {
        'Accuracy':         round(accuracy_score(y_test, y_pred), 4),
        'Precision':        round(precision_score(y_test, y_pred), 4),
        'Recall (attack)':  round(recall_score(y_test, y_pred), 4),
        'F1-score':         round(f1_score(y_test, y_pred), 4),
        'AUC-ROC':          round(roc_auc_score(y_test, y_prob), 4)
    }
    joblib.dump(model, f'data/{name.replace(" ","_")}_baseline.pkl')
    print(f'  -> Recall={results[name]["Recall (attack)"]}, AUC={results[name]["AUC-ROC"]}')

print('\nAll models trained and saved.')

Training Logistic Regression...
  -> Recall=0.4887, AUC=0.9687
Training Random Forest...
  -> Recall=0.8662, AUC=0.9859
Training XGBoost...
  -> Recall=0.8993, AUC=0.9826

All models trained and saved.


In [16]:
table2 = pd.DataFrame(results).T
table2.index.name = 'Model'

print('TABLE 2 — Baseline performance on clean test set')
table2

TABLE 2 — Baseline performance on clean test set


,Accuracy,Precision,Recall (attack),F1-score,AUC-ROC
Model,,,,,
Logistic Regression,0.6501,0.9944,0.4887,0.6553,0.9687
Random Forest,0.9017,0.9879,0.8662,0.9231,0.9859
XGBoost,0.9128,0.9704,0.8993,0.9335,0.9826


In [17]:
joblib.dump(scaler, 'data/scaler.pkl')
print('Scaler saved to data/scaler.pkl')

Scaler saved to data/scaler.pkl
